<a href="https://colab.research.google.com/github/Breezlyx/MineriaDeDatos-Entrega1/blob/main/Entrega_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv("/content/drive/MyDrive/Estudios/Mineria de Datos/job_salary_prediction_dataset.csv")

In [ ]:
df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [ ]:
df.tail()  #visualizamos las ultimas 5 filas

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467
249999,Data Analyst,16,Diploma,2,Technology,Medium,UK,No,5,133084


In [ ]:
df.shape #devuelve la cantidad de filas y columnas respectivamente

(250000, 10)

In [ ]:
df.dtypes #observamos la columna y sus tipos de datos

,0
job_title,object
experience_years,int64
education_level,object
skills_count,int64
industry,object
company_size,object
location,object
remote_work,object
certifications,int64
salary,int64


In [ ]:
df.isnull().sum() #contamos la cantidad de nulos por columna

,0
job_title,0
experience_years,0
education_level,0
skills_count,0
industry,0
company_size,0
location,0
remote_work,0
certifications,0
salary,0


# Limpieza de nulos
Como podemos observar, en nuestro dataset no hay presencia de valores nulos, por lo que no será necesario aplicar técnicas de limpieza en esta fase.

In [ ]:
df.columns #mostramos los nombres de las columnas

Index(['job_title', 'experience_years', 'education_level', 'skills_count',
       'industry', 'company_size', 'location', 'remote_work', 'certifications',
       'salary'],
      dtype='object')

# Mapeo de datos
job_title:	El rol o posición de trabajo (ej: Data Analyst, AI Engineer)

experience_years:	Numero de años de experiencia profesional.

education_level:	Nivel más alto de educación completado.

skills_count:	Número de habilidads técnicas o profesionales.

industry:	Sector de la industria donde pertnece el trabajo.

company_size:	Tamaño de la compañía (small, medium, large)

location	Job: Localización o región

remote_work:	Si es que el trabajo permite trabajo remoto.

certifications:	Número de certificaciones profesionales

salary:	Salario anual de empleados (USD)

In [4]:
df['location'].value_counts().count()

np.int64(10)

In [5]:
df['job_title'].value_counts().count()

np.int64(12)

In [11]:
df['industry'].value_counts().count()

np.int64(10)

In [6]:
df['education_level'].value_counts().count()

np.int64(5)

In [7]:
df['company_size'].value_counts().count()

np.int64(5)

Se realiza un encoding ordinal para las columnas education_level y company_size

In [8]:
# Definimos el diccionario con la jerarquía (de menor a mayor)
education_map = {
    'High School': 1,
    'Diploma': 2,
    'Bachelor': 3,
    'Master': 4,
    'PhD': 5
}

company_map = {
    'Startup': 1,
    'Small': 2,
    'Medium': 3,
    'Large': 4,
    'Enterprise': 5
}

# Aplicamos el mapeo creando nuevas columnas
df['education_encoded'] = df['education_level'].map(education_map)
df['company_encoded'] = df['company_size'].map(company_map)

# Verificamos los cambios
df[['education_level', 'education_encoded', 'company_size', 'company_encoded']].head()

,education_level,education_encoded,company_size,company_encoded
0,Bachelor,3,Medium,3
1,Bachelor,3,Small,2
2,PhD,5,Medium,3
3,PhD,5,Medium,3
4,Bachelor,3,Large,4


Aplicamos One Hot Encoding para la columna remote_work e industry

In [10]:
from sklearn.preprocessing import OneHotEncoder

# Instanciamos el codificador
ohc = OneHotEncoder()

# Ajustamos y transformamos los datos
# El toarray() convierte la matriz dispersa en un arreglo tradicional
ohe = ohc.fit_transform(df[['remote_work']]).toarray()

# Creamos el nuevo DataFrame con los nombres de las categorías
df_onehot = pd.DataFrame(ohe, columns=ohc.get_feature_names_out(['remote_work']))

# Concatenamos el resultado con el DataFrame original
df_encoded = pd.concat([df, df_onehot], axis=1)

# Verificamos
df_encoded.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary,education_encoded,company_encoded,remote_work_Hybrid,remote_work_No,remote_work_Yes
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413,3,3,1.0,0.0,0.0
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764,3,2,0.0,1.0,0.0
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123,5,3,0.0,1.0,0.0
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123,5,3,0.0,0.0,1.0
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069,3,4,0.0,0.0,1.0


Aplicamos Binary Encoder para las columnas job_title y location